# 04 — Explore Transliteration: Urdu Script → Roman Urdu

**Purpose:** Figure out how to convert some of our `urdu-instruct` data from Urdu script to Roman Urdu.

**Why?** Most Pakistanis type in Roman Urdu (Latin script). If the model only understands Urdu script, it won't serve real users. We want ~5-10k Roman Urdu instruction pairs in our training mix.

**Two approaches to explore:**

1. **`Mavkif/Roman-Urdu-Parl-split`** — 6.3M Urdu ↔ Roman Urdu parallel pairs. Can we use it as a lookup/dictionary to transliterate our data?
2. **`urduhack` Python library** — has a transliteration module. Does it work well enough?

We'll test both and see which gives more natural Roman Urdu.

In [1]:
!pip install datasets pandas -q

---
## Part 1: Explore the Mavkif transliteration dataset

### What is this dataset?

6.3 million sentence pairs where each row has the same sentence in:
- **Urdu script** (نستعلیق) 
- **Roman Urdu** (Latin characters)

Think of it as a bilingual dictionary, but for scripts instead of languages. Same language, two writing systems.

### How could we use it?

**Idea:** Take an Urdu script sentence from `urdu-instruct`, look up its words/phrases in this dataset, and construct the Roman Urdu version. Not a perfect approach (language is messy), but worth exploring.

In [2]:
from datasets import load_dataset

print("Loading Mavkif transliteration dataset...")
print("(This is 6.3M rows — might take a minute)")

# Load just a sample first to understand structure
translit = load_dataset("Mavkif/Roman-Urdu-Parl-split", split="train[:1000]")

print(f"\nSample loaded: {len(translit)} rows")
print(f"Columns: {translit.column_names}")
print(f"\nFirst example:")
print(translit[0])

Loading Mavkif transliteration dataset...
(This is 6.3M rows — might take a minute)


README.md: 0.00B [00:00, ?B/s]

train_set.csv:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

validation_set.csv:   0%|          | 0.00/1.83M [00:00<?, ?B/s]

small_validation_set.csv:   0%|          | 0.00/441k [00:00<?, ?B/s]

...

small_test_set.csv:   0%|          | 0.00/447k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6333218 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/20719 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20741 [00:00<?, ? examples/s]


Sample loaded: 1000 rows
Columns: ['Urdu text', 'Roman-Urdu text']

First example:
{'Urdu text': 'وکیل', 'Roman-Urdu text': 'wakeel'}


In [3]:
# Show 10 examples to see the quality of transliteration pairs
import random
random.seed(42)

for i in range(10):
    ex = translit[i]
    print(f"\n--- Example {i+1} ---")
    for col in translit.column_names:
        print(f"  {col}: {str(ex[col])[:300]}")


--- Example 1 ---
  Urdu text: وکیل
  Roman-Urdu text: wakeel

--- Example 2 ---
  Urdu text: وکیل
  Roman-Urdu text: wakeel

--- Example 3 ---
  Urdu text: وکیل ایک ایسی شخصیت کو کہا جاتا ہے کہ جو دوسرے اپنے صارف کی جانب سے یا اسکی بابت گفتگو کرے اس مضمون میں یہ گفتگو قانون سے متعلق تصور کی گئی ہے اور اس وجہ سے یہ مضمون صرف قانونی وکلا کے بارے میں ذکر کرتا ہے
  Roman-Urdu text: wakeel aik aisi shakhsiyat ko kaha jata hai ke jo dosray apne Sarif ki janib se ya uski baabat guftagu kere is mazmoon mein yeh guftagu qanoon se mutaliq tasawwur ki gayi hai aur is wajah se yeh mazmoon sirf qanooni wukla ke baray mein zikar karta hai

--- Example 4 ---
  Urdu text: وکیل قانون
  Roman-Urdu text: wakeel qanoon

--- Example 5 ---
  Urdu text: وکیل قانون ایک شخص جسے دوسرے شخص کی جگہ کام کرنے یا کی نمائندگی کرنے کا اختیار حاصل ہوتا ہے وکیل سفر ایک شخص جو تعطیلات اور سفر کا بندوبست کرتا ہے خفیہ وکیل ایک جاسوس
  Roman-Urdu text: wakeel qanoon 1 shakhs jisay dosray shakhs ki jagah kaam karne ya ki nu

### Quality check

Look at the examples above and ask:
- Is the Roman Urdu natural? (Does it look like how you'd type on WhatsApp?)
- Or is it mechanical transliteration? (Every letter mapped 1:1, nobody actually writes like that)

**This matters because** Roman Urdu has no standard spelling. People write the same word many ways:
- کیا → "kya", "kia", "kiya"
- مجھے → "mujhe", "mujhay", "muje"
- ہے → "hai", "hay", "he"

A good transliteration captures common spellings. A bad one produces robotic output nobody recognizes.

In [4]:
# Let's check: how long are the sentences?
# Short sentences = word-level pairs (useful as dictionary)
# Long sentences = full paragraph transliterations (useful directly)

for col in translit.column_names:
    if isinstance(translit[0][col], str):
        lengths = [len(ex[col]) for ex in translit]
        print(f"'{col}' lengths:")
        print(f"  Min: {min(lengths)}, Max: {max(lengths)}, Avg: {sum(lengths)//len(lengths)}")
        print(f"  <10 chars: {sum(1 for l in lengths if l < 10)}")
        print(f"  >200 chars: {sum(1 for l in lengths if l > 200)}")
        print()

'Urdu text' lengths:
  Min: 2, Max: 251, Avg: 72
  <10 chars: 8
  >200 chars: 35

'Roman-Urdu text' lengths:
  Min: 1, Max: 303, Avg: 86
  <10 chars: 7
  >200 chars: 57



---
## Part 2: Try `urduhack` library

`urduhack` is a Python NLP library for Urdu. It has a transliteration module that converts Urdu script to Roman Urdu programmatically. Let's see if it works.

In [5]:
!pip install urduhack -q

  DEPRECATION: Building 'promise' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'promise'. Discussion can be found at https://github.com/pypa/pip/issues/6334
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
black 25.1.0 requires click>=8.0.0, but you have click 7.1.2 which is incompatible.
celery 5.5.3 requires click<9.0,>=8.1.2, but you have click 7.1.2 which is incompatible.
fiona 1.10.1 requires click~=8.0, but you have click 7.1.2 which is incompatible.
flask 3.1.2 requires click>=8.1.3, but you have click 7.1.2 which is incompatible.
modal 1.4.1 requires click~=8

In [6]:
# Test urduhack transliteration
try:
    from urduhack.urdu_characters import URDU_ALL_CHARACTERS
    from urduhack.normalization import normalize_characters
    # Try transliteration if available
    try:
        from urduhack.transliteration import urdu_to_roman
        HAS_TRANSLIT = True
    except ImportError:
        HAS_TRANSLIT = False
        print("urduhack doesn't have transliteration module.")
        print("This is common — the module was experimental.")
except ImportError:
    print("urduhack not installed or import failed.")
    HAS_TRANSLIT = False

if HAS_TRANSLIT:
    test_sentences = [
        "پاکستان میں موسم کیسا ہے؟",
        "مجھے پائتھون سیکھنی ہے",
        "بریانی کی ریسپی بتائیں",
        "آج کل مہنگائی بہت بڑھ گئی ہے",
    ]
    print("urduhack transliteration results:")
    print("=" * 60)
    for sent in test_sentences:
        roman = urdu_to_roman(sent)
        print(f"Urdu:  {sent}")
        print(f"Roman: {roman}")
        print("-" * 40)
else:
    print("\nSkipping urduhack test — no transliteration available.")
    print("This is fine. We have other approaches.")

urduhack doesn't have transliteration module.
This is common — the module was experimental.

Skipping urduhack test — no transliteration available.
This is fine. We have other approaches.


---
## Part 3: Try a simple character mapping approach

The simplest transliteration: map each Urdu character to its Roman equivalent. This won't be perfect (Urdu has context-dependent pronunciation), but let's see how far it gets.

### Why this is imperfect

Urdu → Roman isn't 1:1. Example:
- 'ع' can be 'a', 'e', 'i', or silent depending on position
- 'ک' is 'k' but 'کھ' is 'kh' (digraph)
- Short vowels aren't written in Urdu script but ARE written in Roman

Still, a basic mapping gives us a starting point to evaluate.

In [7]:
# Basic Urdu → Roman character map
# This is intentionally simplified — not production quality
URDU_TO_ROMAN = {
    'ا': 'a', 'آ': 'aa', 'ب': 'b', 'پ': 'p', 'ت': 't', 'ٹ': 'T',
    'ث': 's', 'ج': 'j', 'چ': 'ch', 'ح': 'h', 'خ': 'kh', 'د': 'd',
    'ڈ': 'D', 'ذ': 'z', 'ر': 'r', 'ڑ': 'R', 'ز': 'z', 'ژ': 'zh',
    'س': 's', 'ش': 'sh', 'ص': 's', 'ض': 'z', 'ط': 't', 'ظ': 'z',
    'ع': 'a', 'غ': 'gh', 'ف': 'f', 'ق': 'q', 'ک': 'k', 'گ': 'g',
    'ل': 'l', 'م': 'm', 'ن': 'n', 'ں': 'n', 'و': 'o', 'ہ': 'h',
    'ھ': 'h', 'ء': '', 'ی': 'i', 'ے': 'e', 'ئ': 'i',
    '۰': '0', '۱': '1', '۲': '2', '۳': '3', '۴': '4',
    '۵': '5', '۶': '6', '۷': '7', '۸': '8', '۹': '9',
    '؟': '?', '۔': '.', '،': ',',
}

def basic_transliterate(text):
    """Naive character-by-character transliteration. Not great, but a baseline."""
    result = []
    for char in text:
        if char in URDU_TO_ROMAN:
            result.append(URDU_TO_ROMAN[char])
        elif char.isascii():
            result.append(char)  # Keep English chars, numbers, punctuation
        elif char == ' ':
            result.append(' ')
        else:
            result.append(char)  # Keep unknown chars as-is
    return ''.join(result)

# Test it
test_sentences = [
    "پاکستان میں موسم کیسا ہے؟",
    "مجھے پائتھون سیکھنی ہے",
    "بریانی کی ریسپی بتائیں",
    "آج کل مہنگائی بہت بڑھ گئی ہے",
]

print("Basic character mapping results:")
print("=" * 60)
for sent in test_sentences:
    roman = basic_transliterate(sent)
    print(f"Urdu:    {sent}")
    print(f"Roman:   {roman}")
    print(f"Natural: (what you'd actually type)")
    print("-" * 40)

print("\nNotice: the basic mapping is robotic. 'pakstan' not 'Pakistan',")
print("'mjhe' not 'mujhe'. Missing vowels that native speakers add.")
print("This is NOT good enough for training data.")

Basic character mapping results:
Urdu:    پاکستان میں موسم کیسا ہے؟
Roman:   pakstan min mosm kisa he?
Natural: (what you'd actually type)
----------------------------------------
Urdu:    مجھے پائتھون سیکھنی ہے
Roman:   mjhe paithon sikhni he
Natural: (what you'd actually type)
----------------------------------------
Urdu:    بریانی کی ریسپی بتائیں
Roman:   briani ki rispi btaiin
Natural: (what you'd actually type)
----------------------------------------
Urdu:    آج کل مہنگائی بہت بڑھ گئی ہے
Roman:   aaj kl mhngaii bht bRh gii he
Natural: (what you'd actually type)
----------------------------------------

Notice: the basic mapping is robotic. 'pakstan' not 'Pakistan',
'mjhe' not 'mujhe'. Missing vowels that native speakers add.
This is NOT good enough for training data.


---
## Part 4: The real solution — LLM-based transliteration

Character mapping fails because Roman Urdu needs **implicit knowledge** about how words are actually spelled by Pakistanis. An LLM already has this knowledge.

### The approach
Take Urdu script text → ask Claude/GPT to rewrite it in Roman Urdu **the way a Pakistani would type it on WhatsApp**.

This is different from mechanical transliteration. The LLM:
- Adds vowels that Urdu script omits
- Uses common spellings ("kya" not "kia")
- Keeps English technical terms in English
- Preserves code-mixing naturally

Let's test this with a few examples using a simple prompt.

**Note:** We're not calling an API here — just showing what the prompt would look like. You can test it manually in ChatGPT/Claude to see the quality before building a pipeline.

In [8]:
# This is the prompt we'd use for LLM-based transliteration
# Test this manually in ChatGPT or Claude to evaluate quality

TRANSLITERATION_PROMPT = """Convert the following Urdu script text to Roman Urdu (Latin script) 
exactly as a young Pakistani would type it on WhatsApp or social media.

Rules:
- Use common Pakistani Roman Urdu spellings (e.g., "kya" not "kia", "hai" not "hay")
- Keep English words in English (don't transliterate technical terms)
- Keep the tone and style natural — informal, the way people actually type
- Do NOT translate — just change the script from Urdu to Roman
- Preserve any numbers, punctuation, and formatting

Urdu text:
{urdu_text}

Roman Urdu:"""

# Example: what the prompt would look like for one example
sample_urdu = "پاکستان میں آج کل مہنگائی بہت بڑھ گئی ہے۔ عام لوگوں کو بہت مشکل ہو رہی ہے۔"

print("PROMPT TO TEST IN ChatGPT/Claude:")
print("=" * 60)
print(TRANSLITERATION_PROMPT.format(urdu_text=sample_urdu))
print("=" * 60)
print("\nExpected output (roughly):")
print("Pakistan mein aaj kal mehngai boht barh gayi hai. Aam logon ko boht mushkil ho rahi hai.")
print("\n→ Copy the prompt above and test it in ChatGPT or Claude.")
print("→ If the output reads naturally, this is our transliteration approach.")

PROMPT TO TEST IN ChatGPT/Claude:
Convert the following Urdu script text to Roman Urdu (Latin script) 
exactly as a young Pakistani would type it on WhatsApp or social media.

Rules:
- Use common Pakistani Roman Urdu spellings (e.g., "kya" not "kia", "hai" not "hay")
- Keep English words in English (don't transliterate technical terms)
- Keep the tone and style natural — informal, the way people actually type
- Do NOT translate — just change the script from Urdu to Roman
- Preserve any numbers, punctuation, and formatting

Urdu text:
پاکستان میں آج کل مہنگائی بہت بڑھ گئی ہے۔ عام لوگوں کو بہت مشکل ہو رہی ہے۔

Roman Urdu:

Expected output (roughly):
Pakistan mein aaj kal mehngai boht barh gayi hai. Aam logon ko boht mushkil ho rahi hai.

→ Copy the prompt above and test it in ChatGPT or Claude.
→ If the output reads naturally, this is our transliteration approach.


---
## Part 5: Test on actual urdu-instruct examples

Let's load a few `urdu-instruct` examples and generate the prompts you'd send to an LLM. This helps you visualize the full pipeline.

In [9]:
# Load urdu-instruct
ui = load_dataset("large-traversaal/urdu-instruct", split="train[:10]")

print(f"Columns: {ui.column_names}")
print(f"\nFirst example:")
for col in ui.column_names:
    print(f"  {col}: {str(ui[0][col])[:200]}")

Columns: ['instruction', 'output', 'category']

First example:
  instruction: صبح کی سیر کے فوائد پر مضمون لکھیں۔
  output: صبح کی سیر انسان کی صحت کے لئے بہت فائدہ مند ہے۔ یہ موٹاپا کم کرتی ہے، دماغ کو تروتازہ کرتی ہے، اور دل کی بیماریوں کے خطرے کو کم کرتی ہے۔ صبح کے تازہ ہوا میں سانس لینے سے انسان کا موڈ بھی خوشگوار رہتا
  category: generation


In [10]:
# Generate transliteration prompts for 3 examples
# You can copy-paste these into ChatGPT/Claude to test quality

for i in range(3):
    ex = ui[i]
    # Combine all text fields for transliteration
    # Adjust column names based on what you saw above
    all_text = "\n".join(
        f"[{col}]: {ex[col]}" 
        for col in ui.column_names 
        if isinstance(ex[col], str) and len(ex[col]) > 0
    )
    
    print(f"\n{'='*60}")
    print(f"EXAMPLE {i+1} — copy this prompt to ChatGPT/Claude:")
    print(f"{'='*60}")
    print(TRANSLITERATION_PROMPT.format(urdu_text=all_text))
    print()


EXAMPLE 1 — copy this prompt to ChatGPT/Claude:
Convert the following Urdu script text to Roman Urdu (Latin script) 
exactly as a young Pakistani would type it on WhatsApp or social media.

Rules:
- Use common Pakistani Roman Urdu spellings (e.g., "kya" not "kia", "hai" not "hay")
- Keep English words in English (don't transliterate technical terms)
- Keep the tone and style natural — informal, the way people actually type
- Do NOT translate — just change the script from Urdu to Roman
- Preserve any numbers, punctuation, and formatting

Urdu text:
[instruction]: صبح کی سیر کے فوائد پر مضمون لکھیں۔
[output]: صبح کی سیر انسان کی صحت کے لئے بہت فائدہ مند ہے۔ یہ موٹاپا کم کرتی ہے، دماغ کو تروتازہ کرتی ہے، اور دل کی بیماریوں کے خطرے کو کم کرتی ہے۔ صبح کے تازہ ہوا میں سانس لینے سے انسان کا موڈ بھی خوشگوار رہتا ہے۔
[category]: generation

Roman Urdu:


EXAMPLE 2 — copy this prompt to ChatGPT/Claude:
Convert the following Urdu script text to Roman Urdu (Latin script) 
exactly as a young Pakis

---
## Part 6: Cost estimate for LLM transliteration

If we use an LLM to transliterate 5,000-10,000 examples, how much would it cost?

In [11]:
# Load more examples to estimate average length
ui_sample = load_dataset("large-traversaal/urdu-instruct", split="train[:1000]")

# Estimate total characters per example
text_cols = [c for c in ui_sample.column_names if isinstance(ui_sample[0][c], str)]
avg_chars = sum(
    sum(len(str(ex[c])) for c in text_cols)
    for ex in ui_sample
) / len(ui_sample)

# Rough token estimate: 1 Urdu char ≈ 1-2 tokens (Urdu tokenizes poorly)
# Plus the prompt template overhead
avg_input_tokens = avg_chars * 1.5 + 200  # text + prompt
avg_output_tokens = avg_chars * 0.8  # Roman Urdu is shorter

print(f"Average chars per example: {avg_chars:.0f}")
print(f"Estimated tokens per example: ~{avg_input_tokens:.0f} input + ~{avg_output_tokens:.0f} output")
print()

# Cost estimates for different providers and volumes
for n_examples in [5000, 10000]:
    total_input = n_examples * avg_input_tokens
    total_output = n_examples * avg_output_tokens
    
    # GPT-4o-mini: $0.15/1M input, $0.60/1M output
    gpt4o_mini_cost = (total_input * 0.15 + total_output * 0.60) / 1_000_000
    
    # Claude Haiku: $0.25/1M input, $1.25/1M output  
    haiku_cost = (total_input * 0.25 + total_output * 1.25) / 1_000_000
    
    # Gemini Flash: $0.075/1M input, $0.30/1M output
    gemini_cost = (total_input * 0.075 + total_output * 0.30) / 1_000_000
    
    print(f"\n--- {n_examples:,} examples ---")
    print(f"  GPT-4o-mini:   ${gpt4o_mini_cost:.2f}")
    print(f"  Claude Haiku:  ${haiku_cost:.2f}")
    print(f"  Gemini Flash:  ${gemini_cost:.2f}")

print("\n→ All options are well within your $80 budget.")
print("→ Even 10k examples would cost just a few dollars.")

Average chars per example: 168
Estimated tokens per example: ~452 input + ~134 output


--- 5,000 examples ---
  GPT-4o-mini:   $0.74
  Claude Haiku:  $1.40
  Gemini Flash:  $0.37

--- 10,000 examples ---
  GPT-4o-mini:   $1.48
  Claude Haiku:  $2.81
  Gemini Flash:  $0.74

→ All options are well within your $80 budget.
→ Even 10k examples would cost just a few dollars.


---
## Summary & Decision

### What we tested

| Approach | Quality | Speed | Cost |
|----------|---------|-------|------|
| Mavkif lookup | Depends on dataset quality | Fast | Free |
| urduhack library | Usually poor | Instant | Free |
| Character mapping | Bad — missing vowels, robotic | Instant | Free |
| LLM transliteration | Best — natural, WhatsApp-style | Minutes | ~$1-5 |

### Recommendation

Use **LLM transliteration** for the Roman Urdu subset. The cost is negligible ($1-5 for 5-10k examples), and the quality is far superior to any automated approach.

### Next steps

1. Test the transliteration prompt on 5-10 examples manually (copy-paste to ChatGPT/Claude)
2. If quality is good, build an automated pipeline with the API
3. Transliterate 5-10k examples from `urdu-instruct`
4. Combine everything into the final dataset